# 🏟️ HaxBall A3 — 3v3 Multi-Agent Shared Policy Training
**Branch:** `MARL` | **Env:** `A3Env` | **Algorithm:** `MaskablePPO` (sb3_contrib)

Architecture: `SharedPolicyVecEnv` — 1 policy drives all 6 agents (3 red + 3 blue).

**Pretrained start:** `a0.1_checkpoints/a0.1_5000000_steps.zip`

## 1. Clone repo

In [ ]:
import os

REPO_URL = "https://github.com/NLightVN/haxball-agent-lite.git"
BRANCH   = "MARL"
REPO_DIR = "/content/haxball-agent-lite"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest...")
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

## 2. Git identity

In [ ]:
GIT_NAME  = "NLightVN"
GIT_EMAIL = "hung24378un@gmail.com"

!git config user.name  "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
print("Git identity configured.")

## 3. Install requirements

In [ ]:
# sb3_contrib cần cho MaskablePPO (A3 dùng action masking)
!pip install -q stable-baselines3[extra] sb3-contrib gymnasium numpy torch

## 4. Mount Google Drive & tạo symlink

**Cấu trúc thư mục trên Drive cần có:**
```
MyDrive/haxball_A3_training/
├── a0.1_checkpoints/
│   └── a0.1_5000000_steps.zip   ← upload file này lên Drive trước khi train lần đầu
└── A3_checkpoints/              ← tự động tạo, checkpoint A3 sẽ lưu vào đây
```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Đường dẫn trên Drive ──────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/haxball_A3_training"
DRIVE_A3_CKPT  = f"{DRIVE_BASE}/A3_checkpoints"    # A3 checkpoints (auto-saved)
DRIVE_A01_CKPT = f"{DRIVE_BASE}/a0.1_checkpoints"  # pretrained A0.1 weights

# Tạo thư mục trên Drive nếu chưa có
!mkdir -p {DRIVE_A3_CKPT}
!mkdir -p {DRIVE_A01_CKPT}

# ── Symlink vào repo ──────────────────────────────────────────────────────────
LOCAL_MODELS = f"{REPO_DIR}/models"
!mkdir -p {LOCAL_MODELS}

# Xóa symlink/thư mục cũ (nếu có)
!rm -rf {LOCAL_MODELS}/A3_checkpoints
!rm -rf {LOCAL_MODELS}/a0.1_checkpoints

# Tạo symlink
!ln -s {DRIVE_A3_CKPT}  {LOCAL_MODELS}/A3_checkpoints
!ln -s {DRIVE_A01_CKPT} {LOCAL_MODELS}/a0.1_checkpoints

print("Symlinks created:")
!ls -la {LOCAL_MODELS}/

## 5. Kiểm tra & cấu hình training

In [ ]:
import os, glob

# ── Cấu hình ─────────────────────────────────────────────────────────────────
# Để resume sau disconnect: set CONTINUE_STEP = số step của checkpoint cuối cùng
# Ví dụ: CONTINUE_STEP = 5_000_000  nếu có a3_5000000_steps.zip trên Drive
# Set = 0 để bắt đầu training mới (sẽ load a0.1_5000000_steps.zip)
CONTINUE_STEP = 0

# ── Kiểm tra checkpoint A3 trên Drive ────────────────────────────────────────
ckpt_dir = f"{REPO_DIR}/models/A3_checkpoints"
ckpts = sorted(glob.glob(f"{ckpt_dir}/a3_*_steps.zip"))
print(f"A3 checkpoints on Drive: {len(ckpts)} found")
for c in ckpts[-5:]:
    print(f"  {os.path.basename(c)}")

# ── Kiểm tra pretrained A0.1 ─────────────────────────────────────────────────
A01_PRETRAINED = f"{REPO_DIR}/models/a0.1_checkpoints/a0.1_5000000_steps.zip"
if os.path.exists(A01_PRETRAINED):
    size_mb = os.path.getsize(A01_PRETRAINED) / 1024 / 1024
    print(f"\nA0.1 pretrained: ✅ Found ({size_mb:.1f} MB)")
    print(f"  → {A01_PRETRAINED}")
else:
    print(f"\nA0.1 pretrained: ❌ NOT FOUND")
    print(f"  → Upload 'a0.1_5000000_steps.zip' to Drive:")
    print(f"     {DRIVE_A01_CKPT}/")
    print("  → Training sẽ bắt đầu từ scratch nếu thiếu file này!")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if CONTINUE_STEP > 0:
    resume_path = f"{ckpt_dir}/a3_{CONTINUE_STEP}_steps.zip"
    exists = os.path.exists(resume_path)
    print(f"Mode: RESUME from step {CONTINUE_STEP:,}")
    print(f"  Checkpoint: {'✅' if exists else '❌ NOT FOUND!'} {os.path.basename(resume_path)}")
else:
    print("Mode: START FRESH (load A0.1 pretrained → fine-tune as A3)")

## 6. Training A3 (3v3 Shared Policy + Self-Play)

In [ ]:
import sys

train_script = f"{REPO_DIR}/training/train_a3.py"
continue_arg = f"--continue_step {CONTINUE_STEP}" if CONTINUE_STEP > 0 else ""

print(f"Running: python {train_script} {continue_arg}")
print("-" * 60)

!{sys.executable} {train_script} {continue_arg}

## 💡 Resume sau khi Colab disconnect

1. **Chạy lại tất cả cells từ đầu** (Clone → Install → Drive → Cfg → Train)
2. Trong cell **Kiểm tra & cấu hình**, xem danh sách checkpoint có sẵn
3. Set `CONTINUE_STEP` = step của checkpoint cuối nhất, ví dụ:
   ```python
   CONTINUE_STEP = 5_000_000  # nếu có a3_5000000_steps.zip
   ```
4. Chạy lại cell **Training** → tự load checkpoint và tiếp tục

**Checkpoint schedule (exponential):**
`500K → 750K → 1.1M → 1.6M → 2.5M → 3.8M → 5.7M → ...`

**Self-play opponent:** Tự động chọn từ A3 checkpoint pool theo temperature annealing.